In [1]:
import cv2
import os
import shutil
import numpy as np
from pathlib import Path
from tqdm import tqdm
import shutil
from ultralytics import YOLO

In [2]:
# --- CONFIGURATION ---
INPUT_IMAGES_DIR = "/home/Joshua/Downloads/leopard_toad_identification/dataset/detect_2/yolo_ready_dataset/images"
INPUT_LABELS_DIR = "/home/Joshua/Downloads/leopard_toad_identification/dataset/detect_2/yolo_ready_dataset//labels"
OUTPUT_DIR = (
    "/home/Joshua/Downloads/leopard_toad_identification/dataset/detect_2/dataset_clahe"
)

# Create output structure
os.makedirs(f"{OUTPUT_DIR}/images", exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/labels", exist_ok=True)


def apply_clahe(image_path, output_path):
    # Read image
    img = cv2.imread(image_path)
    if img is None:
        return

    # Convert to LAB color space (works better than RGB for CLAHE)
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)

    # Apply CLAHE to L-channel (Lightness)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    cl = clahe.apply(l)

    # Merge and convert back to BGR
    limg = cv2.merge((cl, a, b))
    final_img = cv2.cvtColor(limg, cv2.COLOR_LAB2BGR)

    # Save
    cv2.imwrite(output_path, final_img)

In [3]:
image_files = [
    f
    for f in os.listdir(INPUT_IMAGES_DIR)
    if f.lower().endswith((".jpg", ".png", ".jpeg"))
]

print(f"Processing {len(image_files)} images with CLAHE...")

for img_file in tqdm(image_files):
    # 1. Process Image
    src_img = os.path.join(INPUT_IMAGES_DIR, img_file)
    dst_img = os.path.join(OUTPUT_DIR, "images", img_file)
    apply_clahe(src_img, dst_img)

    # 2. Copy corresponding Label (Handle empty files for background images)
    label_name = os.path.splitext(img_file)[0] + ".txt"
    src_label = os.path.join(INPUT_LABELS_DIR, label_name)
    dst_label = os.path.join(OUTPUT_DIR, "labels", label_name)

    if os.path.exists(src_label):
        shutil.copy(src_label, dst_label)
    else:
        # If no label file exists (sometimes implies empty), create an empty one just in case
        open(dst_label, "w").close()

print("Preprocessing complete. Point your YOLO training to 'dataset_clahe'.")

Processing 255 images with CLAHE...


  0%|                                                                                          | 0/255 [00:00<?, ?it/s]

100%|████████████████████████████████████████████████████████████████████████████████| 255/255 [06:01<00:00,  1.42s/it]

Preprocessing complete. Point your YOLO training to 'dataset_clahe'.


In [ ]:
def train_model():
    # 1. Load the model
    model = YOLO(
        "/home/Joshua/Downloads/leopard_toad_identification/detection/runs/detect/Western_Leopard_Toad_Project/yolov8n_clahe_run2/yolov8n.pt"
    )

    # 2. Train the model
    results = model.train(
        data="/home/Joshua/Downloads/leopard_toad_identification/detection/toad_config.yaml",  # Path to your config file
        epochs=100,  # Start with 100 epochs
        imgsz=1280,  # Standard image size
        batch=32,  # Adjust based on your GPU memory
        patience=20,  # Stop if no improvement for 20 epochs
        # --- Augmentation Hyperparameters ---
        # YOLOv8 does Mosaic/MixUp by default. We tune them for camera traps:
        degrees=15.0,  # Rotate +/- 15 degrees
        translate=0.1,  # Shift image slightly
        scale=0.5,  # Scale image (+/- 50%)
        shear=0.0,  # Don't shear (distorts patterns too much)
        perspective=0.0,  # Don't change perspective too much
        flipud=0.0,  # Do NOT flip upside down
        fliplr=0.5,  # Flip Left-Right is okay
        mosaic=1.0,  # Use Mosaic (stitches 4 images together)
        mixup=0.1,  # Mix two images
        # --- Hardware ---
        device="cuda",
        # --- Output ---
        project="/home/Joshua/Downloads/leopard_toad_identification/detection/Western_Leopard_Toad_Project",
        name="yolov8n_clahe_run2",
    )

    success = model.export(format="onnx")

In [10]:
train_model()

New https://pypi.org/project/ultralytics/8.4.21 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.20 🚀 Python-3.12.3 torch-2.10.0+cu128 CUDA:0 (NVIDIA RTX A6000, 48530MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/Joshua/Downloads/leopard_toad_identification/detection/toad_config.yaml, degrees=15.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolov8n.pt, momentum

/home/Joshua/Downloads/leopard_toad_identification/.venv/lib/python3.12/site-packages/torch/onnx/_internal/torchscript_exporter/utils.py:552: OnnxExporterWarning: Exporting to ONNX opset version 22 is not supported. by 'torch.onnx.export()'. The highest opset version supported is 20. To use a newer opset version, consider 'torch.onnx.export(..., dynamo=True)'. 
  _export(


ONNX: slimming with onnxslim 0.1.87...
ONNX: export success ✅ 47.5s, saved as '/home/Joshua/Downloads/leopard_toad_identification/detection/Western_Leopard_Toad_Project/yolov8n_clahe_run2/weights/best.onnx' (12.2 MB)

Export complete (49.1s)
Results saved to /home/Joshua/Downloads/leopard_toad_identification/detection/Western_Leopard_Toad_Project/yolov8n_clahe_run2/weights
Predict:         yolo predict task=detect model=/home/Joshua/Downloads/leopard_toad_identification/detection/Western_Leopard_Toad_Project/yolov8n_clahe_run2/weights/best.onnx imgsz=1280 
Validate:        yolo val task=detect model=/home/Joshua/Downloads/leopard_toad_identification/detection/Western_Leopard_Toad_Project/yolov8n_clahe_run2/weights/best.onnx imgsz=1280 data=/home/Joshua/Downloads/leopard_toad_identification/detection/toad_config.yaml  
Visualize:       https://netron.app
